# Diabetes Risk Prediction: Phase 1 - Load and Explore Data

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-darkgrid')
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

## 1. Load the data

In [ ]:
def load_data(path):
    """Load the CSV at `path` into a DataFrame. Raises FileNotFoundError if it does not exist."""
    if not os.path.exists(path):
        raise FileNotFoundError(f"Dataset not found at: {path}")
    return pd.read_csv(path)

df = load_data('data/diabetes_data.csv')
df.head()

## 2. Shape, data types and missing values

In [ ]:
print(f"Shape: {df.shape}")
df.info()

In [ ]:
missing_values = df.isnull().sum()
print(f"Total missing values: {missing_values.sum()}")

## 3. Target distribution

In [ ]:
target_distribution = df['Diabetes_binary'].value_counts().sort_index()
target_percentage = df['Diabetes_binary'].value_counts(normalize=True).sort_index() * 100

print(f"No Diabetes (0): {target_distribution[0]:,} ({target_percentage[0]:.2f}%)")
print(f"Diabetes (1): {target_distribution[1]:,} ({target_percentage[1]:.2f}%)")

fig, ax = plt.subplots(figsize=(6, 4))
target_distribution.plot(kind='bar', ax=ax, color=['#2ecc71', '#e74c3c'])
ax.set_xlabel('Diabetes Status')
ax.set_ylabel('Count')
ax.set_title('Distribution of Target Variable')
ax.set_xticklabels(['No Diabetes', 'Diabetes'], rotation=0)
plt.tight_layout()
plt.show()

## 4. Statistical summary

In [1]:
df.describe().T

NameError: name 'df' is not defined

> **Understanding statistical summaries** Statistical summaries provide a quick snapshot of your dataset’s structure and variability.  
>
> - *Central tendency:* Metrics like mean and median reveal typical values but can be distorted by skewed data or outliers.  
> - *Spread:* Standard deviation and quartiles highlight how dispersed the data is, where wide spreads may indicate inconsistent scales or high variability.  
> - *Range and extremes:* Min/max values help spot potential data entry errors or outliers needing review.  
> - *Feature type awareness:*  
>   - Continuous features benefit from scale checks and normalization.  
>   - Ordinal or categorical features should be interpreted by the distribution of categories, not numeric statistics.  
>
> *Why this matters:* Understanding the shape, spread, and nature of your data guides preprocessing decisions like scaling, encoding, and handling outliers for reliable modeling.

## 5. Duplicate rows

In [ ]:
feature_cols = [c for c in df.columns if c != 'Diabetes_binary']

repeat_copies = df.duplicated().sum()                      # rows identical to an earlier row
rows_in_dup_groups = df.duplicated(keep=False).sum()       # includes the first occurrence
conflicting_groups = (df.groupby(feature_cols)['Diabetes_binary'].nunique() > 1).sum()

print(f"Repeat copies (first occurrence kept): {repeat_copies}")
print(f"Rows belonging to a duplicate group: {rows_in_dup_groups}")
print(f"Feature-identical groups with conflicting labels: {conflicting_groups}")
print("\nLabels of rows in duplicate groups:")
print(df[df.duplicated(keep=False)]['Diabetes_binary'].value_counts())

## 6. Correlation with the target

In [ ]:
correlations = df.corr()['Diabetes_binary'].sort_values(ascending=False)
print(correlations)

fig, ax = plt.subplots(figsize=(8, 6))
correlations[1:].plot(kind='barh', ax=ax)  # exclude self-correlation
ax.set_xlabel('Correlation Coefficient')
ax.set_title('Feature Correlations with Diabetes Status')
ax.axvline(x=0, color='black', linestyle='--', linewidth=0.8)
plt.tight_layout()
plt.show()

## 7. Observations

- **Shape and types:** 50,000 rows x 22 columns (21 features + `Diabetes_binary`); every column is `int64`, so no type conversion or categorical encoding is needed.
- **No missing values:** `isnull().sum()` is 0 for every column, so no imputation or row removal is needed.
- **Perfectly balanced target:** 25,000 negative and 25,000 positive (50/50). Accuracy is a fair metric here, but recall still matters most clinically because a false negative is a missed case.
- **Very different feature scales:** `BMI` ranges 12-98, `MentHlth`/`PhysHlth` 0-30, `Age` 1-13, `Income` 1-8, and the binary features 0-1. Features must be scaled (e.g. `StandardScaler`, fit on the training set only) before training.
- **Strongest positive correlations with diabetes:** `GenHlth` (0.41), `HighBP` (0.38), `HighChol` (0.28), `BMI` (0.28), `Age` (0.28), `DiffWalk` (0.27).
- **Protective factors (negative correlations):** `Income` (-0.23), `Education` (-0.17), `PhysActivity` (-0.16), `HvyAlcoholConsump` (-0.10). Income and education are socioeconomic proxies, so fairness across groups should be considered.
- **Weak individual features:** `Sex`, `NoDocbcCost` and `AnyHealthcare` have |r| < 0.05; they may add little on their own.
- **Outliers:** `BMI` reaches 98 (mean about 29.9, std about 7.2), and `MentHlth`/`PhysHlth` are heavily skewed toward 0 with a spike at 30.
- **Duplicate rows:** 938 repeat copies (rows identical to an earlier row across all 22 columns); 1,609 rows sit in duplicate groups (largest group: 12 copies). 222 groups have identical features but conflicting labels. Rows in duplicate groups are 1,370 label 0 vs 239 label 1, so dropping them would unbalance the 50/50 target. Decide how to handle them before splitting so copies do not leak across train and test.
- **Multicollinearity:** `HighBP`, `HighChol` and `BMI` correlate with each other, as do `MentHlth` and `PhysHlth`.

# Phase 2 - Preprocessing

In [ ]:
import torch
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

TARGET = 'Diabetes_binary'
BATCH_SIZE = 64

## 8. Duplicate handling decision

Measured on this dataset (seed 42, 60/20/20 stratified split, no deduplication):
- 283 of 10,000 test rows (2.8%) have a feature-identical row in the training set, which would inflate test metrics.
- Dropping the 938 repeat copies leaves 49,062 rows (24,186 negative / 24,876 positive, 49.3% / 50.7%), a small shift from 50/50 that stratified splitting preserves in every split.
- 222 feature-identical pairs carry conflicting labels. Dropping them cannot be justified (we do not know which label is right), so they are kept as irreducible label noise.

Decision: drop exact duplicate rows (all 22 columns) **before** splitting.

In [ ]:
def preprocess_data(dataframe, drop_duplicates=True):
    """Drop exact duplicate rows, split into features/target, then into stratified 60/20/20 train/val/test.

    Returns (X_train, X_test, X_val, y_train, y_test, y_val). Raises ValueError if the DataFrame is empty.
    """
    if dataframe.empty:
        raise ValueError("DataFrame is empty")
    if drop_duplicates:
        dataframe = dataframe.drop_duplicates()

    X = dataframe.drop(columns=[TARGET])
    y = dataframe[TARGET]

    X_train, X_temp, y_train, y_temp = train_test_split(
        X, y, test_size=0.4, stratify=y, random_state=RANDOM_SEED)
    X_val, X_test, y_val, y_test = train_test_split(
        X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=RANDOM_SEED)

    return X_train, X_test, X_val, y_train, y_test, y_val

X_train, X_test, X_val, y_train, y_test, y_val = preprocess_data(df)

In [ ]:
total = len(X_train) + len(X_val) + len(X_test)
for name, y_split in [('train', y_train), ('val', y_val), ('test', y_test)]:
    print(f"{name:5s}: {len(y_split):6d} rows ({len(y_split) / total:.1%}), positive rate {y_split.mean():.2%}")
print(f"full (deduplicated) positive rate: {pd.concat([y_train, y_val, y_test]).mean():.2%}")

In [ ]:
def scale_features(X_train, X_val, X_test):
    """Fit StandardScaler on the training features only; transform train, val and test."""
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val)
    X_test_scaled = scaler.transform(X_test)
    return X_train_scaled, X_val_scaled, X_test_scaled

X_train_scaled, X_val_scaled, X_test_scaled = scale_features(X_train, X_val, X_test)

In [ ]:
X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
X_val_tensor = torch.tensor(X_val_scaled, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.to_numpy(), dtype=torch.long)
y_val_tensor = torch.tensor(y_val.to_numpy(), dtype=torch.long)
y_test_tensor = torch.tensor(y_test.to_numpy(), dtype=torch.long)

In [ ]:
train_loader = DataLoader(TensorDataset(X_train_tensor, y_train_tensor), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(TensorDataset(X_val_tensor, y_val_tensor), batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(TensorDataset(X_test_tensor, y_test_tensor), batch_size=BATCH_SIZE, shuffle=False)

## 9. Validate DataLoader outputs

In [ ]:
X_batch, y_batch = next(iter(train_loader))
print(f"Feature batch shape: {tuple(X_batch.shape)}  dtype={X_batch.dtype}")
print(f"Label batch shape:   {tuple(y_batch.shape)}  dtype={y_batch.dtype}")
print(f"Unique labels in batch: {y_batch.unique().tolist()}")
print(f"Train feature mean (max abs over features): {X_train_tensor.mean(dim=0).abs().max():.2e}")
print(f"Train feature std (min, max over features): {X_train_tensor.std(dim=0).min():.3f}, {X_train_tensor.std(dim=0).max():.3f}")

# Phase 3 - Model Design

In [ ]:
import torch.nn as nn
from torchinfo import summary

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## 10. Architecture

`input (num_features) -> Linear(64) -> ReLU -> Linear(32) -> ReLU -> Linear(1) -> Sigmoid`

- Input size is taken from the data (`X_train_tensor.shape[1]`), not hardcoded.
- Two ReLU hidden layers (64, 32): a modest size for about 29k training rows of 21 tabular features. No regularization is used in the baseline (task 004 constraint): the `dropout` constructor argument defaults to 0.0, and dropout is only introduced as a tuning experiment in Phase 6.
- The output is one sigmoid probability per sample, shape `(batch_size, 1)`, so the loss in Phase 4 should be `nn.BCELoss`.

In [ ]:
class DiabetesClassifier(nn.Module):
    def __init__(self, input_size, hidden_sizes=(64, 32), dropout=0.0):
        super().__init__()
        h1, h2 = hidden_sizes
        layers = [nn.Linear(input_size, h1), nn.ReLU()]
        if dropout > 0:
            layers.append(nn.Dropout(dropout))
        layers += [nn.Linear(h1, h2), nn.ReLU()]
        if dropout > 0:
            layers.append(nn.Dropout(dropout))
        layers += [nn.Linear(h2, 1), nn.Sigmoid()]
        self.network = nn.Sequential(*layers)

    def forward(self, x):
        return self.network(x)

In [ ]:
num_features = X_train_tensor.shape[1]
model = DiabetesClassifier(num_features).to(device)

summary(model, input_size=(BATCH_SIZE, num_features), device=device)

## 11. Forward pass check

In [ ]:
X_batch, y_batch = next(iter(train_loader))
model.eval()
with torch.no_grad():
    output = model(X_batch.to(device))

print(f"Input shape:  {tuple(X_batch.shape)}")
print(f"Output shape: {tuple(output.shape)}")
print(f"Output range: [{output.min():.4f}, {output.max():.4f}]")
print(f"Model device: {next(model.parameters()).device}")

# Phase 4 - Training

## 12. Loss function and optimizer

`nn.BCELoss` (the model already outputs sigmoid probabilities) and `Adam` with `lr=0.001`. The model is re-created after seeding torch so training is reproducible. Labels are `long` tensors, so they are cast to float and reshaped to `(batch, 1)` inside the loop.

In [ ]:
import torch.optim as optim
from typing import List, Tuple

torch.manual_seed(RANDOM_SEED)
model = DiabetesClassifier(num_features).to(device)

criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

print(f"Loss function: {criterion}")
print(f"Optimizer: {optimizer.__class__.__name__}, lr={optimizer.param_groups[0]['lr']}")

## 13. Training function

In [ ]:
from typing import Optional

def train_model(model: nn.Module,
                train_loader: DataLoader,
                val_loader: DataLoader,
                criterion: nn.Module,
                optimizer: optim.Optimizer,
                device: torch.device,
                num_epochs: int = 100,
                print_every: int = 10,
                patience: Optional[int] = None) -> Tuple[int, List[float], List[float]]:
    """Train a PyTorch model with validation monitoring.

    If `patience` is set, training stops after that many epochs without a validation-loss improvement
    (of more than 1e-4) and the best weights are restored; None trains for all `num_epochs`.

    Returns (epochs_run, train_losses, val_losses); the losses hold one value per epoch run.
    """
    train_losses = []
    val_losses = []
    best_val, best_epoch, best_state, wait = float('inf'), 0, None, 0

    for epoch in range(1, num_epochs + 1):
        # Training phase
        model.train()
        running_loss = 0.0
        for X_batch, y_batch in train_loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device).float().unsqueeze(1)

            optimizer.zero_grad()
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * X_batch.size(0)
        train_loss = running_loss / len(train_loader.dataset)

        # Validation phase
        model.eval()
        running_val_loss = 0.0
        correct = 0
        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                X_batch = X_batch.to(device)
                y_batch = y_batch.to(device).float().unsqueeze(1)

                outputs = model(X_batch)
                running_val_loss += criterion(outputs, y_batch).item() * X_batch.size(0)
                correct += ((outputs >= 0.5).float() == y_batch).sum().item()
        val_loss = running_val_loss / len(val_loader.dataset)
        val_accuracy = correct / len(val_loader.dataset)

        train_losses.append(train_loss)
        val_losses.append(val_loss)

        if patience is not None:
            if val_loss < best_val - 1e-4:
                best_val, best_epoch, wait = val_loss, epoch, 0
                best_state = {k: v.clone() for k, v in model.state_dict().items()}
            else:
                wait += 1

        if epoch % print_every == 0 or epoch == 1:
            print(f"Epoch [{epoch}/{num_epochs}] "
                  f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_accuracy:.4f}")

        if patience is not None and wait >= patience:
            print(f"\nEarly stopping at epoch {epoch}: no improvement for {patience} epochs.")
            break

    if patience is not None:
        model.load_state_dict(best_state)
        print(f"Restored best weights from epoch {best_epoch} (Val Loss: {best_val:.4f}).")

    print("\nTraining completed!")
    print(f"Final Train Loss: {train_losses[-1]:.4f}")
    print(f"Final Validation Loss: {val_losses[-1]:.4f}")

    return len(train_losses), train_losses, val_losses

## 14. Train

In [ ]:
num_epochs, train_losses, val_losses = train_model(
    model, train_loader, val_loader, criterion, optimizer, device,
    num_epochs=100, print_every=10)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(range(1, num_epochs + 1), train_losses, label='Train loss')
ax.plot(range(1, num_epochs + 1), val_losses, label='Validation loss')
ax.set_xlabel('Epoch')
ax.set_ylabel('BCE loss')
ax.set_title('Training and validation loss')
ax.legend()
plt.tight_layout()
plt.show()

# Phase 5 - Evaluation

## 15. Evaluation function

Training loss only shows how well the model fits data it has seen. `evaluate_model` runs the trained model on a held-out loader and returns predictions, labels and accuracy / precision / recall / F1 (computed with `sklearn.metrics`).

In [ ]:
from typing import Dict
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

def evaluate_model(model: nn.Module,
                   data_loader: DataLoader,
                   device: torch.device,
                   threshold: float = 0.5) -> Dict:
    """Evaluate a trained model on a dataset and return predictions, labels and metrics.

    Returns a dict with `predictions` (probabilities), `true_labels`, `pred_labels` (binary at `threshold`)
    and `metrics` (accuracy, precision, recall, f1_score).
    """
    model.eval()
    all_predictions = []
    all_labels = []

    with torch.no_grad():
        for X_batch, y_batch in data_loader:
            outputs = model(X_batch.to(device))
            all_predictions.append(outputs.squeeze(1).cpu().numpy())
            all_labels.append(y_batch.cpu().numpy())

    predictions = np.concatenate(all_predictions)
    true_labels = np.concatenate(all_labels)
    pred_labels = (predictions >= threshold).astype(int)

    metrics = {
        'accuracy': accuracy_score(true_labels, pred_labels),
        'precision': precision_score(true_labels, pred_labels, zero_division=0),
        'recall': recall_score(true_labels, pred_labels, zero_division=0),
        'f1_score': f1_score(true_labels, pred_labels, zero_division=0),
    }

    return {
        'predictions': predictions,
        'true_labels': true_labels,
        'pred_labels': pred_labels,
        'metrics': metrics,
    }

## 16. Evaluate on the test set

In [ ]:
test_results = evaluate_model(model, test_loader, device)

for name, value in test_results['metrics'].items():
    print(f"{name:>10}: {value:.4f}")

## 17. Confusion matrix

The confusion matrix shows the types of errors the model makes (false positives vs. false negatives), which makes the precision/recall trade-off easier to interpret.

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, roc_curve, auc

true_labels = test_results['true_labels']
pred_labels = test_results['pred_labels']
predictions = test_results['predictions']

cm = confusion_matrix(true_labels, pred_labels)
tn, fp, fn, tp = cm.ravel()
print(f"True negatives: {tn} | False positives: {fp} | False negatives: {fn} | True positives: {tp}")

disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=[0, 1])
disp.plot(cmap='Blues', colorbar=True)
plt.title('Confusion Matrix - Test Set', fontsize=14, fontweight='bold')
plt.show()

## 18. ROC curve and threshold selection

The ROC curve shows the true positive rate against the false positive rate at every decision threshold, and the AUC summarises how well the model separates the two classes. The default threshold (0.5) is marked together with a lower (0.3) and a higher (0.7) threshold.

In [ ]:
fpr, tpr, thresholds = roc_curve(true_labels, predictions)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(10, 8))
plt.plot(fpr, tpr, color='#3498db', linewidth=2.5, label=f'Model ROC (AUC = {roc_auc:.3f})')
plt.plot([0, 1], [0, 1], color='gray', linestyle='--', linewidth=2, label='Random Classifier (AUC = 0.5)')

# Mark the operating points for the lower, default and higher thresholds
for thr, color in [(0.3, 'green'), (0.5, 'red'), (0.7, 'purple')]:
    idx = np.argmin(np.abs(thresholds - thr))
    plt.scatter(fpr[idx], tpr[idx], color=color, s=150, zorder=5,
                label=f'Threshold {thr}\nTPR={tpr[idx]:.3f}, FPR={fpr[idx]:.3f}')

plt.xlabel('False Positive Rate (1 - Specificity)', fontsize=12, fontweight='bold')
plt.ylabel('True Positive Rate (Sensitivity/Recall)', fontsize=12, fontweight='bold')
plt.title('ROC Curve - Evaluating Threshold Tradeoffs', fontsize=14, fontweight='bold')
plt.legend(loc='lower right', fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\nROC Curve Analysis:")
print("=" * 60)
print(f"AUC Score: {roc_auc:.4f}")
print("Interpretation:")
if roc_auc >= 0.90:
    print("  - Excellent discrimination ability")
elif roc_auc >= 0.80:
    print("  - Good discrimination ability")
elif roc_auc >= 0.70:
    print("  - Fair discrimination ability")
else:
    print("  - Poor discrimination ability")

print("\nMetrics at each threshold:")
for thr in (0.3, 0.5, 0.7):
    m = evaluate_model(model, test_loader, device, threshold=thr)['metrics']
    print(f"  threshold={thr}: " + ", ".join(f"{k}={v:.4f}" for k, v in m.items()))

# Phase 6 - Model Tuning

## 19. Experiment tracking

Every experiment below changes one thing relative to the baseline (`DiabetesClassifier`, 64-32, no regularization, Adam, lr=0.001, 100 epochs) and is tracked with `track_experiment()`. SGD is not used (task constraint).

`evaluate_model()` returns a dict whose numeric scores sit under the `metrics` key, so `track_experiment()` receives `test_results['metrics']` (plus `roc_auc` and the validation F1 as `val_f1_score`).

In [ ]:
import numbers
from typing import Optional

experiment_results = {}

def track_experiment(name: str, 
                     model: nn.Module,
                     train_losses: List[float],
                     val_losses: List[float],
                     test_results: Dict[str, float],
                     notes: str = "") -> None:
    """
    Track experiment results for later comparison.
    
    Args:
        name: Experiment name
        model: Trained model
        train_losses: List of training losses per epoch
        val_losses: List of validation losses per epoch
        test_results: Dictionary from evaluate_model()
        notes: Optional notes about the experiment
    """
   # Keep only numeric metrics so we don't try to tabulate arrays, dicts, etc.
    numeric_metrics = {
        k: float(v) for k, v in test_results.items()
        if isinstance(v, numbers.Number)
    }

    experiment_results[name] = {
        'final_train_loss': float(train_losses[-1]),
        'final_val_loss': float(val_losses[-1]),
        'min_val_loss': float(min(val_losses)),
        'loss_gap': float(abs(train_losses[-1] - val_losses[-1])),
        'metrics': numeric_metrics,           # store all metrics here
        'notes': notes,
        'train_losses': train_losses,         # keep full histories if desired
        'val_losses': val_losses
    }
    print(f"✓ Experiment '{name}' tracked successfully!")


def display_experiment_comparison(sort_by: Optional[str] = "f1",
                                  descending: bool = True) -> Optional[pd.DataFrame]:
    """
    Display a comparison table of all tracked experiments.

    Args:
        sort_by: Metric name to sort by (e.g., 'f1', 'roc_auc', 'accuracy').
                 If None or not present, will fall back to:
                   1) 'roc_auc' if present
                   2) any available metric (alphabetical)
        descending: Sort order for the chosen metric.

    Returns:
        DataFrame with experiment results.
    """
    if not experiment_results:
        print("No experiments tracked yet!")
        return None

    # Collect the union of all metric names across experiments
    all_metric_names = set()
    for res in experiment_results.values():
        all_metric_names.update(res.get('metrics', {}).keys())
    all_metric_names = sorted(all_metric_names)  # stable ordering

    # Determine sort metric
    chosen_sort = None
    if sort_by in all_metric_names:
        chosen_sort = sort_by
    elif all_metric_names:
        chosen_sort = all_metric_names[0]  # fallback to first available
    # If no metrics at all, we’ll sort by final_val_loss

    # Build table rows
    rows = []
    for name, res in experiment_results.items():
        row = {
            'Experiment': name,
            'Val Loss': f"{res['final_val_loss']:.4f}",
            'Loss Gap': f"{res['loss_gap']:.4f}",
        }
        # Add metrics (formatted)
        for m in all_metric_names:
            val = res['metrics'].get(m, None)
            row[m.upper() if m.islower() else m] = (f"{val:.4f}" if isinstance(val, numbers.Number) else "")
        # Also keep raw for sorting
        row['_sort_val'] = (
            res['metrics'].get(chosen_sort)
            if chosen_sort is not None else res['final_val_loss']
        )
        rows.append(row)

    df = pd.DataFrame(rows)

    # Sort
    if chosen_sort is not None:
        df = df.sort_values('_sort_val', ascending=not descending)
    else:
        # No metrics available: sort by Val Loss ascending
        df = df.sort_values('Val Loss', ascending=True, key=lambda s: s.astype(float))

    # Clean up helper column
    df = df.drop(columns=['_sort_val'])

    return df

In [ ]:
def full_metrics(model, test_results):
    """Test metrics from evaluate_model() plus test ROC-AUC and validation F1 (for leakage-free comparison)."""
    metrics = dict(test_results['metrics'])
    metrics['roc_auc'] = roc_auc_score(test_results['true_labels'], test_results['predictions'])
    metrics['val_f1_score'] = evaluate_model(model, val_loader, device)['metrics']['f1_score']
    return metrics

def plot_loss_curves(name, train_losses, val_losses):
    epochs = range(1, len(train_losses) + 1)
    plt.figure(figsize=(7, 3.5))
    plt.plot(epochs, train_losses, label='Train loss')
    plt.plot(epochs, val_losses, label='Validation loss')
    plt.xlabel('Epoch'); plt.ylabel('BCE loss'); plt.title(name); plt.legend()
    plt.tight_layout(); plt.show()

def run_experiment(name, model, lr=0.001, notes="", num_epochs=100, patience=None):
    """Train `model` with Adam (optionally with early stopping), evaluate on val/test, track and plot the loss curves."""
    model = model.to(device)
    n_params = sum(p.numel() for p in model.parameters())
    criterion = nn.BCELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    _, tr_losses, va_losses = train_model(model, train_loader, val_loader, criterion, optimizer, device,
                                          num_epochs=num_epochs, print_every=50, patience=patience)
    results = evaluate_model(model, test_loader, device)
    track_experiment(name, model, tr_losses, va_losses, full_metrics(model, results),
                     notes=f"{notes} | lr={lr}, params={n_params:,}, patience={patience}")
    plot_loss_curves(name, tr_losses, va_losses)
    return model, results

from sklearn.metrics import roc_auc_score

# Track baseline experiment
track_experiment(
    name='Baseline',
    model=model,
    train_losses=train_losses,
    val_losses=val_losses,
    test_results=full_metrics(model, test_results),
    notes='Initial model with default hyperparameters (64-32, no dropout, lr=0.001)'
)

print("\nExperiment tracking system initialized!")
print("Use track_experiment() after training each variation.")
print("Use display_experiment_comparison() to see all results.")

## 20. Experiment 1: Dropout rate

Dropout randomly deactivates neurons during training to reduce overfitting. The baseline shows a train/validation loss gap, so dropout rates of 0.1, 0.2 and 0.3 are tested on the same 64-32 architecture.

In [ ]:
class DiabetesClassifierWithDropout(nn.Module):
    def __init__(self, input_size, hidden_sizes=(64, 32), dropout=0.3):
        super().__init__()
        h1, h2 = hidden_sizes
        self.network = nn.Sequential(
            nn.Linear(input_size, h1),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(h1, h2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(h2, 1),
            nn.Sigmoid(),
        )

    def forward(self, x):
        return self.network(x)

print("Experiment 1: Dropout Tuning")
print("=" * 60)
for rate in (0.1, 0.2, 0.3):
    torch.manual_seed(RANDOM_SEED)
    run_experiment(f"Dropout {rate}", DiabetesClassifierWithDropout(num_features, dropout=rate),
                   notes=f"64-32 with dropout={rate}")

## 21. Experiment 2: Learning rate

The baseline already covers lr=0.001; a lower (0.0001) and a higher (0.01) learning rate are tested with Adam on the baseline architecture.

In [ ]:
print("Experiment 2: Learning Rate Tuning")
print("=" * 60)
for lr in (0.01, 0.0001):
    torch.manual_seed(RANDOM_SEED)
    run_experiment(f"LR {lr}", DiabetesClassifier(num_features), lr=lr, notes="64-32, no dropout")

## 22. Experiment 3: Network architecture

Too small a network underfits; too large a one overfits on ~29k rows. The depth and width are varied (no dropout, lr=0.001): 32, 32-16 and 64-32-16 (the 64-32 baseline is the reference).

In [ ]:
class FlexibleClassifier(nn.Module):
    """MLP with any number of hidden ReLU layers (and optional dropout) for the architecture experiments."""
    def __init__(self, input_size, hidden_sizes, dropout=0.0):
        super().__init__()
        layers, in_features = [], input_size
        for h in hidden_sizes:
            layers += [nn.Linear(in_features, h), nn.ReLU()]
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            in_features = h
        layers += [nn.Linear(in_features, 1), nn.Sigmoid()]
        self.network = nn.Sequential(*layers)

    def forward(self, x):
        return self.network(x)

print("Experiment 3: Architecture Tuning")
print("=" * 60)
for sizes in [(32,), (32, 16), (64, 32, 16)]:
    torch.manual_seed(RANDOM_SEED)
    run_experiment("Arch " + "-".join(map(str, sizes)), FlexibleClassifier(num_features, sizes),
                   notes=f"hidden layers {sizes}, no dropout")

## 23. Compare all experiments

Sorted by test F1-score as the task requires. `Val_F1_Score` (validation F1) is shown alongside, because choosing a configuration on test scores alone leaks the test set into model selection.

In [ ]:
# Display comprehensive comparison of all experiments
print("Comprehensive Experiment Comparison")
print("=" * 70)
print("\nAll experiments sorted by F1-Score (best to worst):\n")

comparison_df = display_experiment_comparison(sort_by='f1_score')
print(comparison_df.to_string(index=False))

# Identify best configuration
best_experiment = comparison_df.iloc[0]['Experiment']
print(f"\n{'=' * 70}")
print(f"Best Configuration: {best_experiment}")
print(f"{'=' * 70}")

print(f"\nKey Insights:")
print("  - Review the 'Loss Gap' column to identify which techniques reduced overfitting")
print("  - Consider the tradeoff between Precision and Recall for your use case")

In [ ]:
print("\nVisualizing Loss Curves for Top 3 Experiments")
print("=" * 60)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
top_3_experiments = comparison_df.head(3)['Experiment'].values

for idx, exp_name in enumerate(top_3_experiments):
    exp_data = experiment_results[exp_name]
    epochs = range(1, len(exp_data['train_losses']) + 1)

    axes[idx].plot(epochs, exp_data['train_losses'], label='Train Loss', linewidth=2, color='#3498db')
    axes[idx].plot(epochs, exp_data['val_losses'], label='Val Loss', linewidth=2, color='#e74c3c')
    axes[idx].set_xlabel('Epoch', fontsize=11)
    axes[idx].set_ylabel('Loss', fontsize=11)
    axes[idx].set_title(f'{exp_name}', fontsize=12, fontweight='bold')
    axes[idx].legend(fontsize=10)
    axes[idx].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 24. Experiment 4: Combined tuning

The best single changes were dropout 0.3, the smaller 32-unit network and the lower learning rate (0.0001). This experiment combines all three: one hidden layer of 32 units, dropout 0.3, Adam with lr=0.0001.

In [ ]:
print("Experiment 4: Combined Tuning")
print("=" * 60)
torch.manual_seed(RANDOM_SEED)
combined_model, combined_results = run_experiment(
    "Combined 32 + Dropout 0.3 + LR 0.0001",
    FlexibleClassifier(num_features, (32,), dropout=0.3),
    lr=0.0001,
    notes="hidden layers (32,), dropout=0.3")

In [ ]:
comparison_df = display_experiment_comparison(sort_by='f1_score')
print(comparison_df.to_string(index=False))

## 25. Reflection on the experiments

All figures come from the experiment comparison table (section 23), test set of 9,813 rows. The train-validation gap is the absolute difference between the final train and validation loss.

### 1. Which experiment performed best overall, and why?

**Dropout 0.3** (64-32 network, lr 0.001) is the best of the dropout, learning-rate and architecture experiments: test F1 0.7660, accuracy 0.7447, recall 0.8238, AUC 0.8178, validation F1 0.7688 (also the highest), and a train-validation gap of only 0.009 (baseline: 0.118).

Best within each family:
- **Dropout:** 0.3 (F1 0.7660), ahead of 0.2 (0.7628) and 0.1 (0.7576).
- **Learning rate:** 0.0001 (F1 0.7620, lowest validation loss of all single experiments, 0.5044).
- **Architecture:** a single hidden layer of 32 units (F1 0.7564, gap 0.014).

Dropout wins because the baseline's main problem was overfitting, and dropout attacks it directly while keeping the highest recall, which matters most for a screening tool. The learning-rate and architecture changes help mainly by overfitting more slowly or having less capacity to overfit.

### 2. Patterns across experiments

- **Dropout reduced the train-validation gap:** 0.118 (baseline) to 0.038 (dropout 0.1), 0.017 (0.2) and 0.009 (0.3). That is a reduction of 68%, 86% and 92%. F1 rose from 0.7366 to 0.7576, 0.7628 and 0.7660.
- **Learning rate:** lr 0.0001 was the most stable. Its validation loss is flat at 0.503-0.504 from epoch 50 to 100 (gap 0.010). With lr 0.001 and 0.01 the model converges in a few epochs, but the validation loss then rises (0.564 and 0.617 at epoch 100) while the training loss keeps falling, giving gaps of 0.118 and 0.158. The best validation losses are within 0.003 of each other (0.5031, 0.5047, 0.5064), so the learning rate changed how quickly the model overfits more than how good it can get. Without early stopping, lr 0.0001 gives the best balance of speed and stability over 100 epochs.
- **Architecture:** wider or deeper networks overfit and did not help. Gap and F1: 32 units 0.014 / 0.7564; 32-16 0.034 / 0.7526; 64-32 (baseline) 0.118 / 0.7366; 64-32-16 0.194 / 0.7310. The deepest model had the worst gap, validation loss (0.6148) and F1 of all.

### 3. Biggest remaining weakness

- **Overfitting:** only the baseline (0.118), lr 0.01 (0.158) and the 64-32-16 network (0.194) have a gap above 0.10. The tuned models are all below 0.04, so overfitting is no longer the main problem.
- **Error type:** for Dropout 0.3 the confusion matrix is TN 3209, FP 1628, FN 877, TP 4099. False positives are more numerous (1628), but the false negatives matter more clinically: 877 of 4976 diabetic patients (17.6%) are missed. For this same model, lowering the decision threshold to 0.3 (Phase 5 threshold analysis) raised recall to 0.93 at the cost of precision (0.66).
- **Performance ceiling:** yes. Validation loss plateaus between 0.503 and 0.506 and accuracy at about 0.75 for every configuration, and AUC stays at 0.78-0.82 ("fair" to "good"). The mission's 95-99% accuracy target is not reachable by tuning this model.

### 4. What was learned about the dataset and problem

- **Feature set:** the 21 survey indicators appear to be the limiting factor. They contain no laboratory values such as blood glucose or HbA1c, which are the strongest predictors of diabetes. This is an inference from the plateau, not something tested here; a feature-engineering or added-feature experiment would be needed to confirm it.
- **Class imbalance:** not a problem in this dataset. The target is balanced 50/50 (section 3 of the data exploration), not 86%/14%, so no experiment was affected by imbalance, and accuracy is a meaningful metric.
- **Diminishing returns:** yes. More capacity made results worse, and combining the best settings (32 units, dropout 0.3, lr 0.0001) gave F1 0.7654, no better than dropout 0.3 alone (0.7660).

**Caveat:** experiments were ranked by test F1, as the task requires, so the winner's test scores are slightly optimistic. The validation F1 column points to the same top group.

## 26. Brainstorm additional improvements

Two techniques are proposed from the list: **Early Stopping** and **Threshold Tuning**. They address the two weaknesses found in the reflection: overfitting during long training runs, and diabetic patients being missed. Class weights and SMOTE are not applicable because this dataset is balanced 50/50. Weight decay, learning-rate scheduling and other activations target the same overfitting problem more indirectly, so they come after early stopping.

### Technique 1: Early Stopping

**1. Problem addressed.** Without a stopping rule, 100 epochs overfit the higher learning rates: the train-validation gap is 0.118 at lr 0.001 and 0.158 at lr 0.01, with final validation losses of 0.564 and 0.617. The same runs reached validation losses of 0.5047 (epoch 9) and 0.5064 (epoch 3) earlier in training. The baseline's F1 of 0.7366 mostly reflects how long it was left running.

**2. Why it is appropriate.** The training loop keeps the weights from the last epoch, not the best epoch. Early stopping tracks the validation loss, stops after `patience` epochs without improvement and restores the best weights. This acts as regularization by preventing the model from fitting noise once it stops generalizing. Selection uses the validation loss only, so the test set is not involved.

**3. Expected results.**
- Validation loss near 0.505 and a gap below 0.02 for the baseline and lr 0.01 settings, which overfit most.
- Estimate (not measured): baseline test F1 of about 0.76-0.77, since models with validation loss 0.505-0.509 scored F1 0.763-0.766. Accuracy about 0.745-0.75.
- It will not break the performance ceiling (about 0.75 accuracy, 0.82 AUC), and models that already generalize well (such as dropout 0.3) would change little.
- Trade-off: `patience` is a new hyperparameter, and a value that is too small can stop during a noisy plateau. Training would stop after roughly 15-30 epochs instead of 100.

**4. Implementation: Medium.** It changes the Phase 4 `train_model` function:
```python
best_val, best_state, wait = float('inf'), None, 0
...  # after computing val_loss each epoch
if val_loss < best_val - 1e-4:
    best_val, wait = val_loss, 0
    best_state = {k: v.clone() for k, v in model.state_dict().items()}
else:
    wait += 1
    if wait >= patience:   # e.g. patience=10
        break
...
model.load_state_dict(best_state)
```
The returned loss lists become shorter than `num_epochs`, so the plotting code should use `len(train_losses)`.

### Technique 2: Threshold Tuning

**1. Problem addressed.** For dropout 0.3 the model misses 877 of 4,976 diabetic patients (17.6%), with recall 0.8238 and precision 0.7157. For screening, a missed patient costs more than an unnecessary follow-up test. The ROC curve (AUC 0.82) shows room to trade false positives for true positives.

**2. Why it is appropriate.** The model outputs a probability, and 0.5 is only a default. Moving the cut-off changes which patients are flagged without retraining. The Phase 5 threshold analysis already measured this for the same model: at 0.3, recall rose from 0.82 to 0.93 and precision fell from 0.72 to 0.66; at 0.7 recall dropped to 0.50. The threshold must be chosen on the validation set, not the test set, to avoid repeating the leakage problem.

**3. Expected results.**
- Recall 0.90-0.93 and precision 0.65-0.68. The ends of this range are measured: at threshold 0.3, recall 0.9297, precision 0.6558, F1 0.7691.
- Accuracy falls by about 3 points to roughly 0.72, which moves further from the accuracy target in the mission.
- F1 stays about the same or improves slightly (0.7691 at 0.3 vs 0.7660 at 0.5).
- Confusion matrix at threshold 0.3 for this model (test set): false negatives fall from 877 to 350 and false positives rise from 1,628 to 2,428 (800 additional false positives).

**4. Implementation: Easy.** `evaluate_model` already accepts a `threshold` argument. The cut-off is chosen on validation predictions:
```python
from sklearn.metrics import precision_recall_curve
val = evaluate_model(model, val_loader, device)
prec, rec, thr = precision_recall_curve(val['true_labels'], val['predictions'])
ok = rec[:-1] >= 0.90                      # recall target set with the clinical team
best_thr = thr[ok][np.argmax(prec[:-1][ok])]
final = evaluate_model(model, test_loader, device, threshold=best_thr)   # test used once
```
The recall target of 0.90 is a clinical decision, not one that can be set from the data alone.

## 27. Test: early stopping in `train_model`

Checks that `patience=None` keeps the old behaviour, that a run with `patience` stops early at exactly `patience` epochs after the last improvement, and that the best weights are restored (the model's validation loss after training equals the best epoch's validation loss).

In [ ]:
def fresh_model(lr=0.001):
    torch.manual_seed(RANDOM_SEED)
    m = DiabetesClassifier(num_features).to(device)
    return m, nn.BCELoss(), torch.optim.Adam(m.parameters(), lr=lr)

def val_loss_of(m):
    m.eval()
    total = 0.0
    with torch.no_grad():
        for X_b, y_b in val_loader:
            out = m(X_b.to(device))
            total += nn.BCELoss()(out, y_b.to(device).float().unsqueeze(1)).item() * X_b.size(0)
    return total / len(val_loader.dataset)

# 1. patience=None trains for every epoch (old behaviour)
m, c, o = fresh_model()
n, tr_ref, va_ref = train_model(m, train_loader, val_loader, c, o, device, num_epochs=15, print_every=100)
assert n == len(tr_ref) == len(va_ref) == 15

# 2. a patience that never triggers gives identical losses
m, c, o = fresh_model()
n, tr, va = train_model(m, train_loader, val_loader, c, o, device, num_epochs=15, print_every=100, patience=1000)
assert n == 15 and np.allclose(tr, tr_ref) and np.allclose(va, va_ref)

# 3. early stopping stops early, exactly `patience` epochs after the last improvement
patience = 5
m, c, o = fresh_model()
n, tr, va = train_model(m, train_loader, val_loader, c, o, device, num_epochs=100, print_every=100, patience=patience)
best_val, best_epoch = float('inf'), 0
for epoch, v in enumerate(va, start=1):
    if v < best_val - 1e-4:
        best_val, best_epoch = v, epoch
assert n == len(tr) == len(va) < 100, f"did not stop early (ran {n} epochs)"
assert n - best_epoch == patience, (n, best_epoch)

# 4. the best weights were restored
restored = val_loss_of(m)
assert abs(restored - va[best_epoch - 1]) < 1e-5, (restored, va[best_epoch - 1])
assert restored <= va[-1], "restored model should not be worse than the last epoch"

print(f"\nAll early-stopping tests passed: stopped at epoch {n}, best epoch {best_epoch}, "
      f"restored val loss {restored:.4f} (last-epoch val loss {va[-1]:.4f}).")

## 28. Part C: Combining techniques

### Proposed experiment: dropout 0.3 + early stopping + validation-chosen threshold

**Configuration**
- **Model:** the 64-32 network with dropout 0.3, Adam at lr 0.001 (the best single-factor model from Phase 6).
- **Training:** `train_model(..., num_epochs=100, patience=10)`, which stops at the best validation loss and restores those weights.
- **Decision rule:** keep the model's probabilities and pick the threshold on the validation set: the highest precision subject to recall >= 0.90. The test set is evaluated once, with that threshold.

### Why this combination

The earlier combined run (32 units + dropout 0.3 + lr 0.0001) gave no gain: F1 0.7654 against 0.7660 for dropout 0.3 alone. All three of its changes fix the same problem, overfitting, so they did not add up; once dropout closed the train-validation gap to 0.009, the learning rate and the smaller network had nothing left to fix.

The three levers here act on different things:
- **Dropout** regularizes what the network learns.
- **Early stopping** controls when training ends, protecting against the overfitting that 100 epochs bring back.
- **Threshold tuning** sets the decision rule, which no amount of training changes. It is also the only lever aimed at the clinical weakness: the 877 missed diabetic patients (17.6%) at threshold 0.5.

### Expected results

These are estimates, not measurements. The ends of the ranges come from measured runs of the dropout 0.3 model.

| Metric | Dropout 0.3 alone (measured) | Combined (expected) |
|---|---|---|
| Validation loss | 0.5086 | about 0.505 |
| AUC | 0.8178 | 0.815-0.822 (unchanged by the threshold) |
| Recall | 0.8238 | 0.90-0.92 |
| Precision | 0.7157 | 0.66-0.69 |
| False negatives | 877 | about 400-500 |
| Accuracy | 0.7447 | about 0.72-0.73 |
| F1 | 0.7660 | about 0.765-0.77 |

- **Recall and precision** are interpolated between two measured points for this model: threshold 0.5 (recall 0.82, precision 0.72) and threshold 0.3 (recall 0.93, precision 0.66).
- **Early stopping** should barely change the metrics, because dropout 0.3 already has a small gap. Its main benefit is stopping at the best validation loss, probably well before 100 epochs, which reduces training time. The exact stopping epoch is not known in advance.
- **Trade-off:** accuracy falls by about 2-3 points and false positives rise, which moves further from the accuracy target in the mission. The clinical team would need to accept that cost in return for catching more patients.
- **Limit:** this will not unlock large gains. All configurations tried so far fall in a narrow band (validation loss 0.503-0.506, AUC about 0.82), which points to a limit in the data rather than in tuning. The one metric expected to move clearly is recall, and it comes from the threshold.

### How to test it fairly

- Choose the threshold on validation predictions only; the 0.90 recall target is a clinical decision.
- Compare against dropout 0.3 alone, adding the components one at a time (dropout, then + early stopping, then + threshold) so each contribution is visible.
- Report false negatives and false positives alongside F1: F1 changes little, while the error counts change a lot.

### Implementation: Medium

`train_model` already supports `patience`; `run_experiment` does not pass it through yet and needs a small change. The threshold step is about five lines (`precision_recall_curve` on the validation set, as sketched in Part B).

## 29. Experiment 5: Dropout 0.3 + early stopping + tuned threshold

The combined experiment proposed in Part C, run as a three-step ablation so each component's contribution is visible:
1. **Reference:** dropout 0.3, 100 epochs, threshold 0.5.
2. **+ early stopping:** `patience=10`, best weights restored, threshold 0.5.
3. **+ tuned threshold:** the threshold is chosen on the *validation* set (highest precision with recall >= 0.90) and the test set is evaluated once with it.

In [ ]:
from sklearn.metrics import precision_recall_curve

def confusion_counts(res):
    tn, fp, fn, tp = confusion_matrix(res['true_labels'], res['pred_labels']).ravel()
    return {'TN': tn, 'FP': fp, 'FN': fn, 'TP': tp}

print("Experiment 5: Dropout 0.3 + early stopping + tuned threshold")
print("=" * 60)

# Step 1 - reference: dropout 0.3, all 100 epochs, threshold 0.5
torch.manual_seed(RANDOM_SEED)
ref_model = DiabetesClassifierWithDropout(num_features, dropout=0.3).to(device)
ref_optimizer = torch.optim.Adam(ref_model.parameters(), lr=0.001)
ref_epochs, _, _ = train_model(ref_model, train_loader, val_loader, nn.BCELoss(), ref_optimizer, device,
                               num_epochs=100, print_every=100)
ref_results = evaluate_model(ref_model, test_loader, device)

# Step 2 - add early stopping (patience 10); best weights are restored
torch.manual_seed(RANDOM_SEED)
es_name = "Dropout 0.3 + early stopping"
es_model, es_results = run_experiment(es_name, DiabetesClassifierWithDropout(num_features, dropout=0.3),
                                      lr=0.001, notes="64-32, dropout 0.3", patience=10)
es_epochs = len(experiment_results[es_name]['train_losses'])

# Step 3 - choose the threshold on the validation set only: highest precision with recall >= 0.90
val_res = evaluate_model(es_model, val_loader, device)
prec, rec, thr = precision_recall_curve(val_res['true_labels'], val_res['predictions'])
ok = rec[:-1] >= 0.90
assert ok.any(), "no threshold reaches the recall target on the validation set"
best_thr = float(thr[ok][np.argmax(prec[:-1][ok])])
val_at_thr = evaluate_model(es_model, val_loader, device, threshold=best_thr)
assert val_at_thr['metrics']['recall'] >= 0.90

tuned_results = evaluate_model(es_model, test_loader, device, threshold=best_thr)   # test set used once with the chosen threshold
tuned_metrics = dict(tuned_results['metrics'])
tuned_metrics['roc_auc'] = roc_auc_score(tuned_results['true_labels'], tuned_results['predictions'])
tuned_metrics['val_f1_score'] = val_at_thr['metrics']['f1_score']
track_experiment(f"{es_name} + threshold", es_model,
                 experiment_results[es_name]['train_losses'], experiment_results[es_name]['val_losses'],
                 tuned_metrics, notes=f"64-32, dropout 0.3, patience=10, threshold={best_thr:.3f} (chosen on validation)")

print(f"\nChosen threshold (validation, recall >= 0.90): {best_thr:.3f}")
print(f"Validation at that threshold: recall={val_at_thr['metrics']['recall']:.4f}, "
      f"precision={val_at_thr['metrics']['precision']:.4f}")

In [ ]:
rows = []
for label, res, epochs, t in [("1. Dropout 0.3 (reference)", ref_results, ref_epochs, 0.5),
                              ("2. + early stopping", es_results, es_epochs, 0.5),
                              ("3. + tuned threshold", tuned_results, es_epochs, best_thr)]:
    m = res['metrics']
    rows.append({'Step': label, 'Epochs run': epochs, 'Threshold': round(t, 3),
                 'Accuracy': round(m['accuracy'], 4), 'Precision': round(m['precision'], 4),
                 'Recall': round(m['recall'], 4), 'F1': round(m['f1_score'], 4), **confusion_counts(res)})
ablation_df = pd.DataFrame(rows)
print(ablation_df.to_string(index=False))
print(f"\nAUC (threshold-independent): reference {roc_auc_score(ref_results['true_labels'], ref_results['predictions']):.4f}, "
      f"early stopping {roc_auc_score(es_results['true_labels'], es_results['predictions']):.4f}")

print("\nAll experiments sorted by F1-Score:\n")
print(display_experiment_comparison(sort_by='f1_score').to_string(index=False))